### Data Collection & Preprocessing

#### Objectives
* Download the dataset programmatically from Kaggle.
* Clean non-image files from the dataset.
* Split the dataset into Train, Validation, and Test sets (70:10:20 ratio).

#### Inputs
* Kaggle Dataset JSON credentials (`kaggle.json`) or Kaggle API token.
* Raw Kaggle dataset: `codeinstitute/cherry-leaves`.

#### Outputs
* Organized image folders inside `inputs/datasets/cherry-leaves/`:
  * `train/` (healthy, powdery_mildew)
  * `validation/` (healthy, powdery_mildew)
  * `test/` (healthy, powdery_mildew)

In [1]:
%pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import shutil
import zipfile

# 1. Setup Kaggle credentials path
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

if os.path.exists("kaggle.json"):
    shutil.copy("kaggle.json", kaggle_dir)

# 2. Set target directories
path_to_data = os.path.join("inputs", "datasets")
os.makedirs(path_to_data, exist_ok=True)

# 3. Download dataset via Kaggle CLI
os.system(f'kaggle datasets download -d codeinstitute/cherry-leaves -p "{path_to_data}"')

# 4. Extract zip archive natively in Python
zip_file_path = os.path.join(path_to_data, "cherry-leaves.zip")
extract_dir = os.path.join(path_to_data, "cherry-leaves")

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    os.remove(zip_file_path)
    print("Download and extraction complete!")

Download and extraction complete!


In [3]:
import os

def clean_non_image_files(data_dir):
    image_extensions = ('.png', '.jpg', '.jpeg')
    non_image_count = 0
    
    for root, dirs, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            if not file.lower().endswith(image_extensions):
                os.remove(file_path)
                non_image_count += 1
                print(f"Removed non-image file: {file_path}")
                
    print(f"Cleaning complete. Total non-image files removed: {non_image_count}")

# Adjust path to account for the nested structure
raw_dataset_dir = 'inputs/datasets/cherry-leaves/cherry-leaves'
clean_non_image_files(raw_dataset_dir)

Cleaning complete. Total non-image files removed: 0


In [ ]:
import os
import random
import shutil

def split_dataset(raw_dir, base_output_dir, train_prop=0.7, val_prop=0.1, test_prop=0.2):
    labels = os.listdir(raw_dir)
    for label in labels:
        label_path = os.path.join(raw_dir, label)
        if not os.path.isdir(label_path):
            continue

        files = [f for f in os.listdir(label_path) if os.path.isfile(os.path.join(label_path, f))]
        random.shuffle(files)

        train_count = int(len(files) * train_prop)
        val_count = int(len(files) * val_prop)

        train_files = files[:train_count]
        val_files = files[train_count:train_count + val_count]
        test_files = files[train_count + val_count:]

        splits = {'train': train_files, 'validation': val_files, 'test': test_files}

        for split, split_files in splits.items():
            split_label_dir = os.path.join(base_output_dir, split, label)
            os.makedirs(split_label_dir, exist_ok=True)

            for file_name in split_files:
                src = os.path.join(label_path, file_name)
                dst = os.path.join(split_label_dir, file_name)
                shutil.copyfile(src, dst)

        print(f"Class '{label}' -> Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

raw_dataset_dir = 'inputs/datasets/cherry-leaves/cherry-leaves'
output_dir = 'inputs/datasets/cherry_leaves_split'
split_dataset(raw_dataset_dir, output_dir)